In [1]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

EARTH_RADIUS_KM = 6371

restaurants = pd.read_csv("/Users/edgardomosesezekielaverilla/Restaurant-Recommendation-System-Redux/data/processed/restaurants_clean.csv")

taxonomy = pd.read_csv("/Users/edgardomosesezekielaverilla/Restaurant-Recommendation-System-Redux/data/reference/category_mapping.csv", encoding="cp1252")

In [2]:
"""Constants"""

CUISINE_WEIGHT = 3.0
TYPE_WEIGHT = 2.0
EXPERIENCE_WEIGHT = 1.5
NUMERIC_WEIGHT = 1.0

SIMILARITY_WEIGHT = 0.80
PROXIMITY_WEIGHT = 0.20
DECAY_FACTOR = 10
SEARCH_RADIUS_KM = 20

In [3]:
selected_restaurant = restaurants[
    (restaurants["name"] == "Sonic Drive-In") &
    (restaurants["city"] == "Ashland City") &
    (restaurants["state"] == "TN")
].iloc[0]

In [4]:
from sklearn.metrics.pairwise import haversine_distances
import numpy as np

def filter_by_radius(
    restaurants,
    selected_restaurant,
    radius_km=20
):
    """
    Return restaurants within a given radius of the
    selected restaurant.
    """

    selected_coords = np.radians([
        [
            selected_restaurant["latitude"],
            selected_restaurant["longitude"]
        ]
    ])

    restaurant_coords = np.radians(
        restaurants[
            ["latitude", "longitude"]
        ]
    )

    distances = haversine_distances(
        selected_coords,
        restaurant_coords
    )[0]

    distances_km = distances * EARTH_RADIUS_KM

    filtered_restaurants = restaurants.copy()

    filtered_restaurants["distance_km"] = distances_km

    filtered_restaurants = filtered_restaurants[
        filtered_restaurants["distance_km"] <= radius_km
    ]

    return filtered_restaurants.reset_index(drop=True)


In [5]:
subset_restaurants = filter_by_radius(
    restaurants=restaurants,
    selected_restaurant=selected_restaurant,
    radius_km=20
)

subset_restaurants = subset_restaurants.sort_values(
    by="distance_km"
).reset_index(drop=True)

subset_restaurants.head(3)

,business_id,name,city,state,postal_code,latitude,longitude,stars,review_count,categories,attributes,hours,is_open,distance_km
0,CF33F8-E6oudUQ46HnavjQ,Sonic Drive-In,Ashland City,TN,37015,36.269593,-87.058943,2.0,6,"Burgers, Fast Food, Sandwiches, Food, Ice Crea...","{'BusinessParking': 'None', 'BusinessAcceptsCr...","{'Monday': '0:0-0:0', 'Tuesday': '6:0-22:0', '...",1,0.000000
1,U_x9aC9UabioGjJb9suzgQ,KFC,Ashland City,TN,37015,36.270909,-87.059858,1.5,6,"Restaurants, Chicken Wings, Fast Food, Chicken...","{'Caters': 'False', 'WiFi': ""'no'"", 'BusinessA...","{'Monday': '10:0-22:0', 'Tuesday': '10:0-22:0'...",1,0.167693
2,ZdJgP7puNRex0Hy4PWIcAQ,McDonald's,Ashland City,TN,37015,36.273119,-87.062329,2.0,14,"Restaurants, Fast Food, Burgers, Coffee & Tea,...","{'Alcohol': ""u'none'"", 'WiFi': ""u'free'"", 'Noi...","{'Monday': '5:0-23:0', 'Tuesday': '5:0-23:0', ...",1,0.495853


In [6]:
restaurants = subset_restaurants.copy()

In [7]:
print(len(restaurants))

71


In [8]:
def calculate_proximity_score(distance_km, decay_factor=10):
    distance_km = np.maximum(distance_km, 0)
    return np.exp(-distance_km / decay_factor)

In [9]:
restaurants["proximity_score"] = calculate_proximity_score(
    restaurants["distance_km"],
    decay_factor=DECAY_FACTOR
)

In [10]:
restaurants[
    ["name", "city", "distance_km"]
].head(10)

,name,city,distance_km
0,Sonic Drive-In,Ashland City,0.000000
1,KFC,Ashland City,0.167693
2,McDonald's,Ashland City,0.495853
3,Marrowbone Creek Brewing,Ashland City,0.520018
4,Gyro City,Ashland City,0.641051
5,El Rey Mexican Restaurant,Ashland City,0.693674
6,Cody's Diner,Ashland City,0.780972
7,Mama D's,Ashland City,0.788658
8,Vuocolo's Italian Pizzeria,Ashland City,0.794609
9,Sunrise Cafe,Ashland City,0.854467


In [11]:
taxonomy_keep = taxonomy[taxonomy["Keep"] == "Yes"]

In [12]:
mapping = taxonomy_keep.set_index("Category").to_dict("index")

In [13]:
categories = [
    c.strip()
    for c in restaurants.iloc[0]["categories"].split(",")
]

In [14]:
for c in categories:
    if c in mapping:
        print(c, "->", mapping[c])

Burgers -> {'Count': 5636, 'Feature Type': 'Restaurant Type', 'Standardized Value': 'Burgers', 'Keep': 'Yes', 'Notes': nan}
Fast Food -> {'Count': 6472, 'Feature Type': 'Restaurant Type', 'Standardized Value': 'Fast Food', 'Keep': 'Yes', 'Notes': nan}
Sandwiches -> {'Count': 8366, 'Feature Type': 'Restaurant Type', 'Standardized Value': 'Sandwiches', 'Keep': 'Yes', 'Notes': nan}
Ice Cream & Frozen Yogurt -> {'Count': 1088, 'Feature Type': 'Restaurant Type', 'Standardized Value': 'Ice Cream & Frozen Yogurt', 'Keep': 'Yes', 'Notes': nan}


In [15]:
def extract_features(category_string, mapping):

    features = {
        "Cuisine": set(),
        "Restaurant Type": set(),
        "Experience": set()
    }

    categories = [
        category.strip()
        for category in category_string.split(",")
    ]

    for category in categories:

        if category in mapping:

            info = mapping[category]

            feature_type = info["Feature Type"]

            value = info["Standardized Value"]

            if feature_type in features and pd.notna(value):
                features[feature_type].add(value)

    return {
        key: list(value)
        for key, value in features.items()
    }

In [16]:
restaurant = restaurants.iloc[0]["categories"]

extract_features(restaurant,mapping)



{'Cuisine': [],
 'Restaurant Type': ['Fast Food',
  'Ice Cream & Frozen Yogurt',
  'Burgers',
  'Sandwiches'],
 'Experience': []}

In [17]:
restaurants["Extracted Features"] = restaurants["categories"].apply(
    lambda x: extract_features(x, mapping)
)

In [18]:
restaurants[["name", "Extracted Features"]].head()

,name,Extracted Features
0,Sonic Drive-In,"{'Cuisine': [], 'Restaurant Type': ['Fast Food..."
1,KFC,"{'Cuisine': [], 'Restaurant Type': ['Fast Food..."
2,McDonald's,"{'Cuisine': [], 'Restaurant Type': ['Fast Food..."
3,Marrowbone Creek Brewing,"{'Cuisine': [], 'Restaurant Type': ['Alcoholic..."
4,Gyro City,"{'Cuisine': ['Greek', 'American', 'Mediterrane..."


In [19]:
restaurants["Restaurant Type"] = restaurants["Extracted Features"].apply(
    lambda features: features["Restaurant Type"]
)

restaurants["Experience"] = restaurants["Extracted Features"].apply(
    lambda features: features["Experience"]
)

restaurants["Cuisine"] = restaurants["Extracted Features"].apply(
    lambda features: features["Cuisine"]
)

In [20]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()

cuisine_mlb = mlb.fit_transform(restaurants["Cuisine"])

cuisine_mlb

array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
       [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
       [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0,

In [21]:
Cuisine = mlb.inverse_transform(cuisine_mlb)

Cuisine

[(),
 (),
 (),
 (),
 ('American', 'Greek', 'Mediterranean'),
 ('Mexican',),
 ('Southern',),
 (),
 ('Italian',),
 ('American',),
 ('Mexican', 'Tex-Mex'),
 (),
 ('Mediterranean',),
 ('American',),
 (),
 ('American',),
 ('Mexican',),
 (),
 ('Chinese',),
 ('Italian',),
 (),
 ('Chinese',),
 (),
 (),
 ('American',),
 (),
 ('Mexican',),
 (),
 ('Italian',),
 ('American',),
 ('Southern',),
 ('American', 'Southern'),
 ('American',),
 ('Japanese',),
 (),
 ('Mexican',),
 ('Japanese',),
 (),
 ('Mexican',),
 (),
 ('American', 'Cajun/Creole'),
 (),
 (),
 (),
 ('Mexican',),
 (),
 ('American', 'Mexican', 'Tex-Mex'),
 (),
 (),
 (),
 ('Italian',),
 (),
 ('Italian',),
 (),
 (),
 ('American', 'Southern'),
 ('Mexican',),
 ('French',),
 (),
 (),
 (),
 ('Mexican',),
 ('American',),
 (),
 (),
 ('Mexican', 'Tex-Mex'),
 ('Mexican',),
 (),
 (),
 ('American',),
 ('Southern',)]

In [22]:
cuisine_df = pd.DataFrame(cuisine_mlb,columns=mlb.classes_)

In [23]:
restaurants["Restaurant Type"].apply(type).value_counts()

Restaurant Type
<class 'list'>    71
Name: count, dtype: int64

In [24]:
all_types = set()

for lst in restaurants["Restaurant Type"]:
    all_types.update(type(x) for x in lst)

all_types

{str}

In [25]:
restaurants[
    restaurants["Restaurant Type"].apply(
        lambda lst: any(not isinstance(x, str) for x in lst)
    )
][["name", "Restaurant Type"]]

,name,Restaurant Type


In [26]:
restaurant_type_mlb = mlb.fit_transform(restaurants["Restaurant Type"])

restaurant_type = mlb.inverse_transform(restaurant_type_mlb)

restaurant_type_df = pd.DataFrame(restaurant_type_mlb, columns=mlb.classes_)

restaurant_type_df

,Alcoholic Beverages,Barbeque,Beer,Breakfast & Brunch,Buffets,Burgers,Cafe,Chicken Wings,Comfort Food,Desserts,...,Ice Cream & Frozen Yogurt,Pizza,Salad,Sandwiches,Seafood,Soul Food,Sushi,Tacos,Tapas/Small Plates,Waffles
0,0,0,0,0,0,1,0,0,0,0,...,1,0,0,1,0,0,0,0,0,0
1,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,0,1,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
67,0,0,0,1,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
68,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
69,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0


In [27]:
experience_mlb = mlb.fit_transform(restaurants["Experience"])

experience = mlb.inverse_transform(experience_mlb)

experience_df = pd.DataFrame(experience_mlb, columns=mlb.classes_)

experience_df

,Beer,Wine
0,0,0
1,0,0
2,0,0
3,1,0
4,0,0
...,...,...
66,0,0
67,0,0
68,0,0
69,0,0


In [28]:
numeric_features_df = restaurants[
    [
        "latitude",
        "longitude",
        "stars",
        "review_count"
    ]
]

In [29]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

scaled_numeric = scaler.fit_transform(numeric_features_df[["stars","review_count"]])

scaled_numeric

scaled_numeric_df = pd.DataFrame(scaled_numeric, columns = ['stars', 'review_count'])

scaled_numeric_df


,stars,review_count
0,-1.468348,-0.732786
1,-1.979391,-0.732786
2,-1.468348,-0.396309
3,1.597908,-0.522488
4,1.086865,-0.774846
...,...,...
66,1.086865,0.234586
67,1.597908,-0.732786
68,1.086865,2.716106
69,0.064780,-0.606607


In [30]:
weighted_cuisine = cuisine_df * CUISINE_WEIGHT

weighted_type = restaurant_type_df * TYPE_WEIGHT

weighted_experience = experience_df * EXPERIENCE_WEIGHT

weighted_numeric = scaled_numeric_df * NUMERIC_WEIGHT

In [31]:
recommendation_feature_matrix = pd.concat(
    [
        weighted_cuisine,
        weighted_type,
        weighted_experience,
        weighted_numeric
    ],
    axis=1
)

In [32]:
def find_restaurant(
    restaurant_name,
    city,
    restaurants
):

    selected_index = restaurants[
        (restaurants["name"] == restaurant_name) &
        (restaurants["city"] == city)
    ].index[0]

    return selected_index

In [33]:
def get_feature_vector(
    selected_index,
    recommendation_feature_matrix
):

    selected_vector = recommendation_feature_matrix.loc[
        [selected_index]
    ]

    return selected_vector

In [34]:

def calculate_similarity(
    selected_vector,
    recommendation_feature_matrix
):

    similarity_scores = cosine_similarity(
        selected_vector,
        recommendation_feature_matrix
    )

    similarity_df = pd.DataFrame(
        similarity_scores.T,
        columns=["similarity"]
    )

    return similarity_df

In [35]:
def get_top_recommendations(
    similarity_df,
    restaurants,
    selected_index,
    top_n=10,
    exclude_same_chain=False
):

    filtered_df = similarity_df.drop(selected_index)

    filtered_df = filtered_df.copy()

    filtered_df["proximity_score"] = restaurants.loc[
        filtered_df.index,
        "proximity_score"
    ]

    if exclude_same_chain:
        selected_name = restaurants.loc[selected_index, "name"]

        filtered_df = filtered_df[
            restaurants.loc[filtered_df.index, "name"] != selected_name
        ]

    filtered_df["hybrid_score"] = (
        SIMILARITY_WEIGHT * filtered_df["similarity"]
        + PROXIMITY_WEIGHT * filtered_df["proximity_score"]
    )

    top_similarity_df = filtered_df.nlargest(
        top_n,
        "hybrid_score"
    )

    return top_similarity_df

In [36]:
def format_recommendations(
    top_similarity_df,
    restaurants
):

    recommendation_table = restaurants.loc[
        top_similarity_df.index
    ]

    recommendation_table = recommendation_table[
        [
            "name",
            "city",
            "state",
            "stars",
            "review_count",
            "distance_km",
            "proximity_score"
        ]
    ]

    recommendation_table = pd.concat(
        [recommendation_table, top_similarity_df],
        axis=1
    )

    return recommendation_table

In [37]:
def recommend_restaurants(
    restaurant_name,
    city,
    restaurants,
    recommendation_feature_matrix,
    top_n=10
):

    selected_index = find_restaurant(
        restaurant_name,
        city,
        restaurants
    )

    selected_vector = get_feature_vector(
        selected_index,
        recommendation_feature_matrix
    )

    similarity_df = calculate_similarity(
        selected_vector,
        recommendation_feature_matrix
    )

    top_similarity_df = get_top_recommendations(
    similarity_df=similarity_df,
    restaurants=restaurants,
    selected_index=selected_index,
    top_n=top_n,
    exclude_same_chain=True
    )   

    recommendation_table = format_recommendations(
        top_similarity_df,
        restaurants
    )

    return recommendation_table

In [38]:
recommend_restaurants(
    restaurant_name="Sonic Drive-In",
    city="Ashland City",
    restaurants=restaurants,
    recommendation_feature_matrix=recommendation_feature_matrix,
    top_n=10
)

,name,city,state,stars,review_count,distance_km,proximity_score,similarity,proximity_score,hybrid_score
20,Burger King,Ashland City,TN,1.0,9,3.167642,0.728503,0.733264,0.728503,0.732312
59,Dairy Queen Grill & Chill,Joelton,TN,3.0,6,18.023494,0.164911,0.854988,0.164911,0.716973
2,McDonald's,Ashland City,TN,2.0,14,0.495853,0.951624,0.638649,0.951624,0.701244
47,Burger King,Pleasant View,TN,2.0,11,14.549055,0.233422,0.754804,0.233422,0.650528
43,Sonic,Pleasant View,TN,3.0,5,14.436271,0.236070,0.746180,0.236070,0.644158
49,Wendy's,Pleasant View,TN,2.5,16,14.731565,0.229201,0.742215,0.229201,0.639612
1,KFC,Ashland City,TN,1.5,6,0.167693,0.983371,0.487822,0.983371,0.586931
58,Subway,Joelton,TN,3.5,5,18.016861,0.165020,0.668061,0.165020,0.567453
48,Hardee's,Pleasant View,TN,2.0,9,14.606050,0.232096,0.643348,0.232096,0.561097
51,Subway,Pleasant View,TN,4.0,7,15.630376,0.209499,0.596995,0.209499,0.519495


### Verify proximity score calculation

In [39]:
restaurants[
    [
        "name",
        "distance_km",
        "proximity_score"
    ]
].head(15)

,name,distance_km,proximity_score
0,Sonic Drive-In,0.000000,1.000000
1,KFC,0.167693,0.983371
2,McDonald's,0.495853,0.951624
3,Marrowbone Creek Brewing,0.520018,0.949327
4,Gyro City,0.641051,0.937906
5,El Rey Mexican Restaurant,0.693674,0.932984
6,Cody's Diner,0.780972,0.924875
7,Mama D's,0.788658,0.924164
8,Vuocolo's Italian Pizzeria,0.794609,0.923614
9,Sunrise Cafe,0.854467,0.918102
